In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")


# Load

In [2]:
# path dataset di Kaggle
PATH = "/kaggle/input/playground-series-s5e12/"

train = pd.read_csv(PATH + "train.csv")
test  = pd.read_csv(PATH + "test.csv")
sample_sub = pd.read_csv(PATH + "sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)

train.head()

Train shape: (700000, 26)
Test shape : (300000, 25)


,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


# Cek target & missing values

In [3]:
TARGET = "diagnosed_diabetes"
ID_COL  = "id"

print(train[TARGET].value_counts(normalize=True))
print("\nMissing values (train):")
print(train.isna().sum().sort_values(ascending=False).head(20))

print("\nMissing values (test):")
print(test.isna().sum().sort_values(ascending=False).head(20))


diagnosed_diabetes
1.0    0.623296
0.0    0.376704
Name: proportion, dtype: float64

Missing values (train):
id                                    0
age                                   0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                     0
hdl_cholesterol                       0
ldl_cholesterol                       0
triglycerides                         0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
dtype: int64

Missing values (test):
id                                    0
age           

# Feature & Preprocessing

In [4]:
# daftar fitur (semua kolom kecuali id dan target)
features = [c for c in train.columns if c not in [TARGET, ID_COL]]

X = train[features].copy()
X_test = test[features].copy()
y = train[TARGET].astype(int)

# deteksi kolom kategori dan numerik
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = [c for c in features if c not in cat_cols]

print("Jumlah fitur total   :", len(features))
print("Fitur numerik        :", len(num_cols))
print("Fitur kategorikal    :", len(cat_cols))

# 1) Tangani missing value untuk numerik
if len(num_cols) > 0:
    for c in num_cols:
        median_val = X[c].median()
        X[c] = X[c].fillna(median_val)
        X_test[c] = X_test[c].fillna(median_val)

# 2) Tangani missing value untuk kategori (kalau ada)
if len(cat_cols) > 0:
    for c in cat_cols:
        X[c] = X[c].astype(str).fillna("missing")
        X_test[c] = X_test[c].astype(str).fillna("missing")
    
    # Ordinal encode kategori → angka
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])
    X_test[cat_cols] = oe.transform(X_test[cat_cols])
else:
    oe = None

print("Shape X     :", X.shape)
print("Shape X_test:", X_test.shape)


Jumlah fitur total   : 24
Fitur numerik        : 18
Fitur kategorikal    : 6
Shape X     : (700000, 24)
Shape X_test: (300000, 24)


# Setup CV & array OOF

In [5]:
N_FOLDS = 10
RANDOM_STATE = 42

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# array untuk OOF prediction
oof_lgb = np.zeros(len(X))
oof_xgb = np.zeros(len(X))
oof_cat = np.zeros(len(X))

# array untuk pred test
pred_lgb = np.zeros(len(X_test))
pred_xgb = np.zeros(len(X_test))
pred_cat = np.zeros(len(X_test))


# Loop train

In [6]:
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== FOLD {fold} / {N_FOLDS} ==========")
    
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # -----------------------------
    # LightGBM
    # -----------------------------
    lgb_model = LGBMClassifier(
        n_estimators=4000,
        learning_rate=0.02,
        objective="binary",
        subsample=0.8,
        colsample_bytree=0.8,
        max_depth=-1,
        num_leaves=32,
        reg_alpha=0.5,
        reg_lambda=0.5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[]
    )

    val_pred_lgb = lgb_model.predict_proba(X_val)[:, 1]
    test_pred_lgb = lgb_model.predict_proba(X_test)[:, 1]

    oof_lgb[val_idx] = val_pred_lgb
    pred_lgb += test_pred_lgb / N_FOLDS

    print("LGBM AUC:", roc_auc_score(y_val, val_pred_lgb))

    # -----------------------------
    # XGBoost
    # -----------------------------
    xgb_model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    val_pred_xgb = xgb_model.predict_proba(X_val)[:, 1]
    test_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]

    oof_xgb[val_idx] = val_pred_xgb
    pred_xgb += test_pred_xgb / N_FOLDS

    print("XGB  AUC:", roc_auc_score(y_val, val_pred_xgb))

    # -----------------------------
    # CatBoost
    # -----------------------------
    cat_model = CatBoostClassifier(
        depth=6,
        learning_rate=0.03,
        n_estimators=2500,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE,
        verbose=False
    )

    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val)
    )

    val_pred_cat = cat_model.predict_proba(X_val)[:, 1]
    test_pred_cat = cat_model.predict_proba(X_test)[:, 1]

    oof_cat[val_idx] = val_pred_cat
    pred_cat += test_pred_cat / N_FOLDS

    print("CAT  AUC:", roc_auc_score(y_val, val_pred_cat))



========== FOLD 1 / 10 ==========
[LightGBM] [Info] Number of positive: 392676, number of negative: 237324
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043338 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1643
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623295 -> initscore=0.503559
[LightGBM] [Info] Start training from score 0.503559
LGBM AUC: 0.7281508460036593
XGB  AUC: 0.7266893562458487
CAT  AUC: 0.7267010428445118

========== FOLD 2 / 10 ==========
[LightGBM] [Info] Number of positive: 392676, number of negative: 237324
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.037743 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_c

# OOF score & simple blend

In [7]:
print("=== OOF AUC per model ===")
auc_lgb = roc_auc_score(y, oof_lgb)
auc_xgb = roc_auc_score(y, oof_xgb)
auc_cat = roc_auc_score(y, oof_cat)

print(f"LGBM : {auc_lgb:.6f}")
print(f"XGB  : {auc_xgb:.6f}")
print(f"CAT  : {auc_cat:.6f}")

# weight bisa kamu tuning
w_lgb = 0.4
w_xgb = 0.3
w_cat = 0.3

oof_blend = w_lgb * oof_lgb + w_xgb * oof_xgb + w_cat * oof_cat
auc_blend = roc_auc_score(y, oof_blend)

print("\n=== BLEND OOF AUC ===")
print(f"Blend (LGB {w_lgb}, XGB {w_xgb}, CAT {w_cat}) : {auc_blend:.6f}")


=== OOF AUC per model ===
LGBM : 0.728168
XGB  : 0.727265
CAT  : 0.726751

=== BLEND OOF AUC ===
Blend (LGB 0.4, XGB 0.3, CAT 0.3) : 0.728305


In [8]:
oof_df = pd.DataFrame({
    ID_COL: train[ID_COL],
    TARGET: y,
    "oof_lgb": oof_lgb,
    "oof_xgb": oof_xgb,
    "oof_cat": oof_cat,
    "oof_blend": oof_blend
})

oof_df.to_csv("oof_base_models.csv", index=False)
oof_df.head()


,id,diagnosed_diabetes,oof_lgb,oof_xgb,oof_cat,oof_blend
0,0,1,0.441125,0.494169,0.460141,0.462743
1,1,1,0.580614,0.596888,0.586645,0.587305
2,2,0,0.210620,0.209341,0.230118,0.216086
3,3,1,0.509307,0.507807,0.524169,0.513316
4,4,1,0.807705,0.796942,0.770931,0.793444


In [9]:
# pred test untuk blend sama seperti OOF
pred_blend = w_lgb * pred_lgb + w_xgb * pred_xgb + w_cat * pred_cat

sub_blend = sample_sub.copy()
sub_blend[TARGET] = pred_blend

sub_blend.to_csv("submission_blend_base.csv", index=False)
sub_blend.head()


,id,diagnosed_diabetes
0,700000,0.490436
1,700001,0.680667
2,700002,0.777023
3,700003,0.390291
4,700004,0.924149


# Stacking logistic regression

In [10]:
X_meta = oof_df[["oof_lgb", "oof_xgb", "oof_cat"]]
y_meta = oof_df[TARGET]

meta_lr = LogisticRegression(
    max_iter=1000,
    n_jobs=-1
)

meta_lr.fit(X_meta, y_meta)

oof_stack = meta_lr.predict_proba(X_meta)[:, 1]
auc_stack = roc_auc_score(y_meta, oof_stack)

print("AUC stacking (LR di atas 3 model):", auc_stack)


AUC stacking (LR di atas 3 model): 0.7283582131648004


In [11]:
# susun fitur meta untuk test (pakai pred test masing2 model)
X_meta_test = pd.DataFrame({
    "oof_lgb": pred_lgb,
    "oof_xgb": pred_xgb,
    "oof_cat": pred_cat
})

pred_stack = meta_lr.predict_proba(X_meta_test)[:, 1]

sub_stack = sample_sub.copy()
sub_stack[TARGET] = pred_stack

sub_stack.to_csv("submission_stack_lr.csv", index=False)
sub_stack.head()

,id,diagnosed_diabetes
0,700000,0.482199
1,700001,0.706323
2,700002,0.794730
3,700003,0.357449
4,700004,0.888696
